# 🔬 Unified Multi-Modal Pipeline Experiment
**TransNetV2 $ightarrow$ Adaptive Keyframes $ightarrow$ SigLIP-2 $ightarrow$ OCR $ightarrow$ SAM $ightarrow$ DAM-3B**

This notebook runs the complete end-to-end multi-modal frame extraction and enrichment pipeline on a target video and renders inline visual inspection cards.

In [ ]:
# 1. Pull Latest Repository
import os
from pathlib import Path

REPO_DIR = Path("/kaggle/working/AIC-2026")
if not REPO_DIR.exists():
    !git clone --depth=1 -b feature/dam-text-extraction https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git /kaggle/working/AIC-2026
else:
    %cd /kaggle/working/AIC-2026
    !git pull origin feature/dam-text-extraction

%cd /kaggle/working/AIC-2026


In [ ]:
# 2. Check GPU Environment & Install Unified Dependencies
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

!pip install --quiet easyocr
!pip install --quiet git+https://github.com/facebookresearch/segment-anything.git
!pip install --quiet --no-deps git+https://github.com/NVlabs/describe-anything.git@153ad3d33c29324e9197f565547c6bc8500da02d
!pip install --quiet huggingface_hub accelerate einops addict yapf

print("✓ All dependencies installed successfully!")


In [ ]:
# 3. Run Unified Multi-Modal Pipeline Experiment
TARGET_VIDEO_ID = "L21_V003"
MAX_FRAMES_TO_TEST = 5
OUTPUT_DIR = "/kaggle/working/multimodal_results"

!python scripts/run_unified_multimodal_experiment.py \n    --video-id {TARGET_VIDEO_ID} \n    --max-frames {MAX_FRAMES_TO_TEST} \n    --score-threshold 0.30 \n    --max-regions 3 \n    --max-words 50 \n    --output-dir {OUTPUT_DIR}


In [ ]:
# 4. Display Visual Inspection Cards & Inspect Generated Manifests
import json
from pathlib import Path
from IPython.display import Image as IPImage, display

out_p = Path("/kaggle/working/multimodal_results")
map_csv_p = out_p / "map-keyframes" / f"{TARGET_VIDEO_ID}.csv"
jsonl_p = out_p / "unified" / f"{TARGET_VIDEO_ID}.jsonl"
vis_dir = out_p / "visualizations" / TARGET_VIDEO_ID

print("=" * 75)
if map_csv_p.exists():
    print(f"📄 Generated Canonical Map-Keyframes CSV ({map_csv_p.name}):")
    print(map_csv_p.read_text().strip())
    print("-" * 75)

if jsonl_p.exists():
    records = [json.loads(line) for line in jsonl_p.read_text().splitlines() if line.strip()]
    print(f"📦 Generated {len(records)} Unified Frame Records in JSONL:")
    if records:
        sample = records[0]
        print(f"  • Frame UID:       {sample["frame_uid"]}")
        print(f"  • Keyframe #:      {sample["keyframe_n"]} (idx: {sample["frame_idx"]}, pts: {sample["pts_time_s"]}s)")
        print(f"  • SigLIP-2 Vector: Dim {len(sample.get("siglip_embedding") or [])}")
        print(f"  • OCR Text:        {sample["ocr"]["full_text"]!r}")
        print(f"  • DAM Captions:    {len(sample["dam_descriptions"])} objects")
        for d in sample["dam_descriptions"]:
            print(f"      - [{d["class_label"]}]: "{d["caption_en"]}" ({d["word_count"]} words)")
    print("=" * 75)

card_files = sorted(vis_dir.glob("*.jpg"))
print(f"
🖼️ Displaying {len(card_files)} Visual Inspection Cards:
")
for card in card_files:
    print(f"📌 Card: {card.name}")
    display(IPImage(filename=str(card)))
    print("-" * 75)
